# JIT Compilation

torchode is compatible with PyTorch's JIT compiler `torch.compile`. By compiling the solver together with your model, you can speed up training as well as inference, because the compiler removes the overhead of the Python interpreter and fuses many small operations, such as the ones in an ODE solver, into larger kernels.

Note that `torch.compile` applied to a module only compiles its `forward` method. torchode's solvers are called through `solve`, so you have to compile that method directly.

Let's begin by importing everything we need in this example.

In [1]:
import timeit

import torch
import torch.nn as nn

import torchode as to

torch.random.manual_seed(180819023);

Now we define a simple neural ODE given by an MLP with two hidden layers.

In [2]:
class Model(nn.Module):
    def __init__(self, n_features, n_hidden):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_features, n_hidden),
            nn.Softplus(),
            nn.Linear(n_hidden, n_hidden),
            nn.Softplus(),
            nn.Linear(n_hidden, n_features),
        )

    def forward(self, t, y):
        return self.layers(y)


n_features = 5
model = Model(n_features=n_features, n_hidden=32)

Next, we construct the solver components and put them together into the solver `AutoDiffAdjoint`, which computes the parameter derivatives by backpropagating through the solver.

In [3]:
dev = torch.device("cpu")
term = to.ODETerm(model)
step_method = to.Dopri5(term=term)
step_size_controller = to.IntegralController(atol=1e-6, rtol=1e-3, term=term)
solver = to.AutoDiffAdjoint(step_method, step_size_controller).to(dev)

Then we compile the `solve` method. The actual compilation happens lazily on the first call and takes a few seconds.

In [4]:
compiled_solve = torch.compile(solver.solve)

As a last step, we combine the initial condition `y0` and the evaluation points `t_eval` into a problem instance.

In [5]:
batch_size = 3
t_eval = torch.tile(torch.linspace(0.0, 3.0, 10), (batch_size, 1))
problem = to.InitialValueProblem(
    y0=torch.zeros((batch_size, n_features)).to(dev), t_eval=t_eval.to(dev)
)

Here we see that both the normal and the compiled solver get the same stats and approximately the same result. The results are not identical, because the compiler reorders and fuses operations, which changes the floating point rounding.

In [6]:
sol = solver.solve(problem)
sol_compiled = compiled_solve(problem)

print(sol.stats)
print(sol_compiled.stats)
print("Max absolute difference", (sol.ys - sol_compiled.ys).abs().max().item())

{'n_f_evals': tensor([38, 38, 38]), 'n_steps': tensor([6, 6, 6]), 'n_accepted': tensor([6, 6, 6]), 'n_initialized': tensor([10, 10, 10])}
{'n_f_evals': tensor([38, 38, 38]), 'n_steps': tensor([6, 6, 6]), 'n_accepted': tensor([6, 6, 6]), 'n_initialized': tensor([10, 10, 10])}
Max absolute difference 2.5033950805664062e-06


And finally we can compare the two in terms of runtime. We call the compiled solver a few more times before measuring, so that we do not measure the compilation.

In [7]:
for _ in range(3):
    compiled_solve(problem)

n = 100
eager_time = timeit.timeit(lambda: solver.solve(problem), number=n) / n
compiled_time = timeit.timeit(lambda: compiled_solve(problem), number=n) / n
print(f"Eager:    {eager_time * 1e3:.2f} ms per solve")
print(f"Compiled: {compiled_time * 1e3:.2f} ms per solve")

Eager:    1.99 ms per solve
Compiled: 1.28 ms per solve
